# Who absorbed the DDEP loss? Analysis notebook
Run the cells in order (Shift+Enter). Put `ghana_banking_panel_2020_2025.xlsx` in the same folder as this notebook. Results are saved as CSV files in an `out` folder.

In [5]:
# Run once if packages are missing (remove the # to use):
# %pip install pandas numpy scipy statsmodels openpyxl

In [6]:
DATA = 'ghana_banking_panel_2020_2025.xlsx'   # change the path if the file is elsewhere
print(DATA)

ghana_banking_panel_2020_2025.xlsx


## Load the data, build the sample and the measures

In [7]:
# Who absorbed the DDEP loss? Analysis script (Python 3; pandas, numpy, scipy, statsmodels).
# Writes tables to ./out/. Seed fixed for permutation and bootstrap results.
import numpy as np,pandas as pd,statsmodels.api as sm
from scipy import stats
import sys,os
def load():
        return pd.read_excel(DATA,sheet_name='Panel_Long')
os.makedirs('out',exist_ok=True)
rng=np.random.default_rng(20261110)
P=load()
P['burden']=1-P.cir-P.pbt_margin
W=P.pivot(index='bank_code',columns='year')
def g(v,y): return W[(v,y)]
banks=W.index
# sample rule
base_year=pd.Series(2021,index=banks)
have=lambda b,y: not np.isnan(g('roa',y)[b]) and not np.isnan(g('pbt_margin',y)[b]) and not np.isnan(g('cir',y)[b])
def sample(bymap=None):
    rows=[]
    for b in banks:
        by=2021 if have(b,2021) else (2020 if have(b,2020) else None)
        if by and have(b,2022) and have(b,2025): rows.append((b,by))
    return rows
S=sample()
print(len(S),[b for b,_ in S if _!=2021], 'excluded:',[b for b in banks if b not in [x for x,_ in S]])
def build(basefun):
    R=[]
    for b in banks:
        by=basefun(b)
        if by is None or not (have(b,2022) and have(b,2025)): continue
        r=dict(bank=b)
        for v in ['roa','pbt_margin','cir','nim','burden']:
            r[v+'_base']=by(v) if callable(by) else None
        R.append(r)
    return R
def frame(mode='2021'):
    R=[]
    for b in banks:
        if not (have(b,2022) and have(b,2025)): continue
        if mode=='2021': ys=[2021] if have(b,2021) else ([2020] if have(b,2020) else None)
        elif mode=='2020': ys=[2020] if have(b,2020) else None
        elif mode=='avg':
            ys=[y for y in (2020,2021) if have(b,y)] or None
        elif mode=='strict2021': ys=[2021] if have(b,2021) else None
        if not ys: continue
        r=dict(bank=b,base_years='+'.join(map(str,ys)))
        for v in ['roa','pbt_margin','cir','nim','burden']:
            r[v+'_base']=np.mean([g(v,y)[b] for y in ys])
            r[v+'_22']=g(v,2022)[b]; r[v+'_25']=g(v,2025)[b]
        r['ln_size']=P[(P.bank_code==b)&(P.year==2020)].ln_assets_share.iloc[0]
        R.append(r)
    D=pd.DataFrame(R).set_index('bank')
    D['excess']=D.burden_22-D.burden_base
    for v in ['roa','pbt_margin']:
        D[v+'_rebound']=D[v+'_25']-D[v+'_22']
        D[v+'_scar']=D[v+'_25']-D[v+'_base']
        D[v+'_shock']=D[v+'_22']-D[v+'_base']
        D[v+'_recovered']=(D[v+'_25']>=D[v+'_base']).astype(int)
    return D
D=frame('2021'); D.to_csv('out/bank_measures.csv')
print(D.shape)
def ols_perm(x,y,B=20000):
    x=np.asarray(x,float); y=np.asarray(y,float)
    X=sm.add_constant(x); m=sm.OLS(y,X).fit(); h=m.get_robustcov_results('HC3')
    b=m.params[1]
    yy=np.array(y); xx=np.array(x)
    cnt=0
    for _ in range(B):
        yp=rng.permutation(yy); bp=np.polyfit(xx,yp,1)[0]
        if abs(bp)>=abs(b)-1e-12: cnt+=1
    return dict(n=len(x),slope=b,se_hc3=h.bse[1],ci_lo=h.conf_int()[1][0],ci_hi=h.conf_int()[1][1],p_hc3=h.pvalues[1],p_perm=(cnt+1)/(B+1),r2=m.rsquared)
def spear_boot(x,y,B=10000):
    rho,p=stats.spearmanr(x,y); n=len(x); bs=[]
    x=np.array(x);y=np.array(y)
    for _ in range(B):
        i=rng.integers(0,n,n)
        if len(set(x[i]))<2 or len(set(y[i]))<2: continue
        bs.append(stats.spearmanr(x[i],y[i])[0])
    return dict(n=n,rho=rho,p=p,lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5))
res={}

21 ['UMB'] excluded: ['NIB', 'PBL']
(21, 26)


## Step 1. Distribution of the 2022 burden

In [8]:
# 1 descriptives
d=D[['burden_base','burden_22','excess']].describe(percentiles=[.25,.5,.75]).T
print(d.round(3)); d.to_csv('out/t1_burden_describe.csv')
print('rank by excess:\n',D.excess.sort_values(ascending=False).round(3).to_string())
for nm,x in [('baseline ROA','roa_base'),('ln size','ln_size')]:
    r=spear_boot(D[x],D.excess,2000); print('Spearman excess vs',nm,r)

             count   mean    std    min    25%    50%    75%    max
burden_base   21.0  0.043  0.126 -0.463  0.027  0.054  0.088  0.213
burden_22     21.0  0.873  0.479  0.360  0.555  0.708  1.103  2.229
excess        21.0  0.830  0.480  0.303  0.444  0.600  1.094  2.141
rank by excess:
 bank
CBG      2.141
CAL      1.762
FNB      1.220
OBL      1.180
UMB      1.129
ZBL      1.094
ABSA     0.978
ABG      0.975
SCB      0.966
FBL      0.848
ADB      0.600
GCB      0.577
GTB      0.548
SBG      0.528
EBG      0.492
FBN      0.444
RBL      0.434
BOA      0.427
FABL     0.406
UBA      0.383
SG-GH    0.303
Spearman excess vs baseline ROA {'n': 21, 'rho': np.float64(-0.3898480664276356), 'p': np.float64(0.08063680392623304), 'lo': np.float64(-0.6991230776640667), 'hi': np.float64(0.11329441983304736)}
Spearman excess vs ln size {'n': 21, 'rho': np.float64(0.15799753280787843), 'p': np.float64(0.4939646804960469), 'lo': np.float64(-0.3577059782563098), 'hi': np.float64(0.5724012705410241)}


## Step 2. Rebound, scarring and H1

In [9]:
# 2
print('recovered ROA',D.roa_recovered.sum(),'of',len(D),' PBT',D.pbt_margin_recovered.sum())
print(D[['roa_base','roa_22','roa_25','roa_rebound','roa_scar','pbt_margin_base','pbt_margin_22','pbt_margin_25','pbt_margin_scar']].describe().round(4).T)
H1={}
for out_ in ['roa','pbt_margin']:
    H1[out_+'_scar']=ols_perm(D.excess,D[out_+'_scar'])
    H1[out_+'_rebound']=ols_perm(D.excess,D[out_+'_rebound'])
    r=stats.spearmanr(D.excess,D[out_+'_scar']); H1[out_+'_scar_spearman']=dict(rho=r[0],p=r[1])
H1df=pd.DataFrame(H1).T; print(H1df.round(4)); H1df.to_csv('out/t2_H1.csv')

recovered ROA 17 of 21  PBT 15
                 count    mean     std    min    25%    50%    75%    max
roa_base          21.0  0.0257  0.0162 -0.005  0.019  0.026  0.034  0.063
roa_22            21.0 -0.0296  0.0426 -0.142 -0.043 -0.026  0.000  0.022
roa_25            21.0  0.0295  0.0147  0.000  0.020  0.031  0.039  0.056
roa_rebound       21.0  0.0591  0.0378 -0.018  0.032  0.061  0.077  0.142
roa_scar          21.0  0.0038  0.0143 -0.034  0.002  0.008  0.012  0.025
pbt_margin_base   21.0  0.3930  0.2240 -0.127  0.331  0.446  0.509  0.736
pbt_margin_22     21.0 -0.4058  0.6312 -2.119 -0.563 -0.272  0.038  0.225
pbt_margin_25     21.0  0.4864  0.1993  0.012  0.424  0.541  0.584  0.802
pbt_margin_scar   21.0  0.0934  0.2301 -0.381 -0.022  0.093  0.142  0.680
                             n   slope  se_hc3   ci_lo   ci_hi   p_hc3  \
roa_scar                  21.0  0.0025  0.0083 -0.0148  0.0197  0.7695   
roa_rebound               21.0  0.0683  0.0078  0.0519  0.0846  0.0000   
roa_sca

## Step 3. Decomposition of the margin change

In [10]:
# 3 decomposition (pbt margin = 1 - cir - burden)
D['d_pbt_25_22']=D.pbt_margin_25-D.pbt_margin_22
D['d_cir_25_22']=-(D.cir_25-D.cir_22); D['d_burden_25_22']=-(D.burden_25-D.burden_22)
D['d_pbt_25_b']=D.pbt_margin_25-D.pbt_margin_base
D['d_cir_25_b']=-(D.cir_25-D.cir_base); D['d_burden_25_b']=-(D.burden_25-D.burden_base)
D['d_nim_25_22']=D.nim_25-D.nim_22; D['d_nim_25_b']=D.nim_25-D.nim_base
assert np.allclose(D.d_pbt_25_22,D.d_cir_25_22+D.d_burden_25_22)
assert np.allclose(D.d_pbt_25_b,D.d_cir_25_b+D.d_burden_25_b)
dec=D[['d_pbt_25_22','d_cir_25_22','d_burden_25_22','d_nim_25_22','d_pbt_25_b','d_cir_25_b','d_burden_25_b','d_nim_25_b']]
print(dec.describe().round(3).T); dec.to_csv('out/t3_decomposition.csv')
share=(D.d_burden_25_22/D.d_pbt_25_22)
print('share of rebound from burden fall: median',share.median(), 'banks where burden>cost:',(D.d_burden_25_22.abs()>D.d_cir_25_22.abs()).sum())
print('baseline: burden contrib>0 count',(D.d_burden_25_b>0).sum(),'cost contrib>0',(D.d_cir_25_b>0).sum())
r=stats.spearmanr(D.d_nim_25_b,D.roa_scar); print('NIM change vs ROA scar spearman',r)

                count   mean    std    min    25%    50%    75%    max
d_pbt_25_22      21.0  0.892  0.574 -0.097  0.539  0.830  1.066  2.131
d_cir_25_22      21.0  0.009  0.233 -0.550 -0.130  0.020  0.080  0.550
d_burden_25_22   21.0  0.883  0.532  0.369  0.516  0.686  1.076  2.103
d_nim_25_22      21.0 -0.000  0.022 -0.055 -0.014  0.000  0.014  0.035
d_pbt_25_b       21.0  0.093  0.230 -0.381 -0.022  0.093  0.142  0.680
d_cir_25_b       21.0  0.040  0.282 -0.390 -0.070  0.040  0.110  1.070
d_burden_25_b    21.0  0.053  0.211 -0.494 -0.018  0.021  0.080  0.640
d_nim_25_b       21.0 -0.001  0.027 -0.077 -0.010 -0.006  0.013  0.045
share of rebound from burden fall: median 0.9779492833517089 banks where burden>cost: 20
baseline: burden contrib>0 count 15 cost contrib>0 11
NIM change vs ROA scar spearman SignificanceResult(statistic=np.float64(0.33279169778267803), pvalue=np.float64(0.14046629918079465))


## Step 4. Rank persistence (H2)

In [11]:
# 4 H2
H2=spear_boot(D.roa_base,D.roa_25); print('H2',H2)
H2b=spear_boot(D.pbt_margin_base,D.pbt_margin_25); print('H2 pbt',H2b)
q=lambda s: pd.qcut(s.rank(method='first'),4,labels=['Q1','Q2','Q3','Q4'])
tt=pd.crosstab(q(-D.roa_base),q(-D.roa_25),rownames=['baseline ROA quartile'],colnames=['2025 ROA quartile']); print(tt); tt.to_csv('out/t4_transition.csv')

H2 {'n': 21, 'rho': np.float64(0.607364252383067), 'p': np.float64(0.0034991187412650388), 'lo': np.float64(0.10850784511174871), 'hi': np.float64(0.9641581300379891)}
H2 pbt {'n': 21, 'rho': np.float64(0.41636897291793445), 'p': np.float64(0.06044975182565557), 'lo': np.float64(-0.06831039384763847), 'hi': np.float64(0.8079848685612689)}
2025 ROA quartile      Q1  Q2  Q3  Q4
baseline ROA quartile                
Q1                      4   0   0   2
Q2                      2   3   0   0
Q3                      0   2   3   0
Q4                      0   0   2   3


## Step 5. Robustness checks

In [13]:
# robustness
rob=[]
rob.append(dict(check='Main (baseline 2021; UMB 2020)',**{k:H1['roa_scar'][k] for k in ['n','slope','p_perm']},rho=H2['rho'],rho_lo=H2['lo'],rho_hi=H2['hi']))
for mode,label in [('2020','Baseline 2020 only'),('avg','Baseline 2020-21 average'),('strict2021','Excluding UMB (2021 only)')]:
    Dm=frame(mode); r1=ols_perm(Dm.excess,Dm.roa_scar,5000); r2=spear_boot(Dm.roa_base,Dm.roa_25,3000)
    rob.append(dict(check=label,n=r1['n'],slope=r1['slope'],p_perm=r1['p_perm'],rho=r2['rho'],rho_lo=r2['lo'],rho_hi=r2['hi']))
r1=H1['pbt_margin_scar']; rob.append(dict(check='Outcome: PBT margin',n=r1['n'],slope=r1['slope'],p_perm=r1['p_perm'],rho=H2b['rho'],rho_lo=H2b['lo'],rho_hi=H2b['hi']))
loo=[]
for b in D.index:
    Dm=D.drop(b); r1=ols_perm(Dm.excess,Dm.roa_scar,2000); r2=stats.spearmanr(Dm.roa_base,Dm.roa_25)
    loo.append(dict(dropped=b,slope=r1['slope'],p_perm=r1['p_perm'],p_hc3=r1['p_hc3'],rho=r2[0]))
loo=pd.DataFrame(loo); loo.to_csv('out/t6_leave_one_out.csv')
print(loo.round(4).to_string())
rob.append(dict(check='Leave-one-out: slope range',n=len(D)-1,slope=f"{loo.slope.min():.3f} to {loo.slope.max():.3f}",p_perm=f"max {loo.p_perm.max():.3f}",rho=f"{loo.rho.min():.2f} to {loo.rho.max():.2f}",rho_lo=None,rho_hi=None))
rb=pd.DataFrame(rob); print(rb.to_string()); rb.to_csv('out/t5_robustness.csv',index=False)
D.to_csv('out/bank_measures_full.csv')

   dropped   slope  p_perm   p_hc3     rho
0      ABG  0.0035  0.5532  0.7195  0.7779
1     ABSA  0.0022  0.7526  0.7923  0.5615
2      ADB  0.0027  0.7166  0.7505  0.5940
3      BOA  0.0029  0.6822  0.7397  0.5972
4      CAL  0.0027  0.7136  0.8352  0.5972
5      CBG  0.0093  0.2949  0.2918  0.5600
6      EBG  0.0028  0.6917  0.7499  0.5985
7     FABL  0.0022  0.7491  0.8020  0.6015
8      FBL  0.0024  0.7176  0.7696  0.5975
9      FBN -0.0009  0.8711  0.8914  0.7527
10     FNB  0.0012  0.8616  0.8758  0.6032
11     GCB  0.0028  0.6997  0.7440  0.5985
12     GTB  0.0013  0.8516  0.8766  0.5494
13     OBL  0.0008  0.9070  0.9143  0.6055
14     RBL  0.0024  0.7586  0.7846  0.5940
15     SBG  0.0033  0.6272  0.7057  0.6025
16     SCB  0.0024  0.7521  0.7749  0.5601
17   SG-GH  0.0030  0.6712  0.7431  0.5980
18     UBA  0.0031  0.6692  0.7293  0.5975
19     UMB  0.0019  0.7976  0.8161  0.6008
20     ZBL  0.0020  0.7916  0.8099  0.6015
                            check   n            slope